In [1]:
import os

# Même correction que les Sessions 1 à 4 : VSCode connecté à un serveur
# Jupyter distant démarre parfois le kernel avec le mauvais dossier de
# travail. Cette cellule force le bon cwd avant tout.
if os.path.basename(os.getcwd()) != "notebooks":
    os.chdir("/home/jovyan/notebooks")

print("cwd:", os.getcwd())

cwd: /home/jovyan/notebooks


# ClimaCity Paris -- Session 5
## Machine Learning avec MLlib, optimisation et bilan d'architecture

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Prérequis** : Avoir complété les Jours 1 et 2 -- la table Delta
`data/output/delta/disponibilite` doit être présente.

---

Ce notebook couvre l'intégralité du Jour 3 du projet ClimaCity Paris.

- **Partie 1 -- Matin (3h30)** : Machine Learning distribué avec MLlib.
  Construction de features, clustering des stations (K-Means), modèle de
  prédiction du taux d'occupation (GBTRegressor), validation croisée,
  et suivi des expériences avec MLflow.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne.  
> Les cellules `# [CORRECTION]` proposent une solution.


---
> **Transparence sur l'usage d'un assistant IA**
>
> Plusieurs cellules de ce notebook (indiquées individuellement par un
> commentaire à la première personne, ex. *"J'ai adapté..."*, *"J'ai corrigé..."*)
> ont été complétées avec l'aide de l'assistant IA **Claude / Claude Code**
> (Anthropic), après que nous avons constaté que certaines cellules d'exercice
> reposaient sur des données ou variables absentes du projet réel (schéma
> différent du jeu de données réellement fourni, plage de dates 2022-2023 de
> l'énoncé alors que nos données couvrent 2020-2021, fichiers "pré-préparés"
> jamais générés par aucune cellule).
>
> Chaque correction a été vérifiée par rapport aux données réelles (schéma,
> plage de dates, comptages) avant d'être acceptée -- voir la section
> *« Transparence sur l'utilisation d'un assistant IA »* du `README.md` du
> projet pour le détail complet de la démarche et de ce qui reste de notre
> propre analyse (choix de conception, interprétation des résultats).
---


---
## Section 0 -- Configuration


In [2]:
from pathlib import Path
import time, warnings
warnings.filterwarnings("ignore")

# ── Chemins ──────────────────────────────────────────────────────────────────
DATA_DIR         = Path("../data")
OUTPUT_DIR       = DATA_DIR / "output"
DELTA_DISPONIBLE = OUTPUT_DIR / "delta" / "disponibilite"
# J'ai corrigé ce chemin : "stations_info.csv" n'existe pas dans nos données.
# Le fichier disponible est le CSV brut Vélib (une ligne par snapshot), que
# j'utiliserai en 1.4 pour récupérer les coordonnées GPS des stations.
STATIONS_CSV      = DATA_DIR / "velib" / "raw" / "historique_stations.csv"
MODELS_DIR        = OUTPUT_DIR / "models"
MLFLOW_DIR         = OUTPUT_DIR / "mlruns"

for p in [DELTA_DISPONIBLE]:
    assert p.exists(), f"Fichier manquant : {p} -- relancez les Jours 1 et 2"
for p in [MODELS_DIR, MLFLOW_DIR]:
    p.mkdir(parents=True, exist_ok=True)

APP_NAME      = "ClimaCity-Paris-Jour3"
SHUFFLE_PARTS = 8
SEED          = 42

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.driver.memory", "6g")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- http://localhost:4040")


Spark 3.5.3 -- http://localhost:4040


In [4]:
# Chargement de la table Delta consolidée
df = (
    spark.read.format("delta").load(str(DELTA_DISPONIBLE))
)
df.cache()
df.count()   # force le cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()

Table consolidée : 10,922,018 lignes  |  18 colonnes
root
 |-- nom_station: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- horodatage: timestamp (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- velos_total: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- statut: string (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: double (nullable = true)
 |-- vent_kmh: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)



---
# PARTIE 1 -- Apprentissage automatique avec MLlib

## 1.1 Les trois abstractions fondamentales de MLlib

MLlib repose sur trois interfaces qui permettent de construire des pipelines
ML reproductibles et composables.

### Transformer

Un `Transformer` prend un DataFrame en entrée et retourne un DataFrame enrichi.
Il implémente la méthode `transform(df)`.  
Exemples : `VectorAssembler`, `StandardScaler` (après fit), `Binarizer`.

### Estimator

Un `Estimator` est un algorithme qui s'entraîne sur des données.
Il implémente la méthode `fit(df)` et retourne un `Transformer` (le modèle ajusté).  
Exemples : `GBTRegressor`, `KMeans`, `StandardScaler` (avant fit).

### Pipeline

Un `Pipeline` enchaîne une liste ordonnée de `Transformer` et d'`Estimator`.
Lorsqu'on appelle `pipeline.fit(df)`, chaque étape est exécutée en séquence.
Le résultat est un `PipelineModel`, qui lui-même est un `Transformer`.

```
DataFrame d'entrée
      │
      ▼
  Étape 1 : VectorAssembler  (Transformer)  → features brutes
      │
      ▼
  Étape 2 : StandardScaler   (Estimator)    → calcule mean/std sur train
      │         ↓ fit → StandardScalerModel (Transformer)
      ▼
  Étape 3 : GBTRegressor     (Estimator)    → entraîne le modèle
                ↓ fit → GBTRegressionModel  (Transformer)
      │
      ▼
DataFrame de sortie (avec colonne "prediction")
```

La clé du Pipeline : on appelle `fit()` **une seule fois** sur les données
d'entraînement. Le `PipelineModel` résultant peut être appliqué sur les données
de test **sans risque de fuite d'information** entre train et test.


---
## 1.2 Construction des features

Un bon modèle commence par de bonnes features. Nous allons construire
un vecteur de features à partir des informations disponibles :
contexte temporel, conditions météorologiques, et historique récent de la station.


In [5]:
from pyspark.sql.functions import (
    col, when, sin, cos, lit, lag, lead, coalesce, avg as spark_avg,
    to_timestamp, unix_timestamp, round as spark_round,
    dayofyear
)
from pyspark.sql.window import Window

# ── 1. Features temporelles cycliques ────────────────────────────────────────
# L'heure 23 et l'heure 0 sont proches dans le temps mais éloignées en valeur
# brute. On encode l'heure et le jour de semaine sur un cercle (sin/cos)
# pour que le modèle perçoive leur nature cyclique
#
# Chaque information ('heure', 'jour_sem', 'mois') est encodé par un sinus + un cosinus
# Le cercle est divisé en parts égales proportionnelles au nombre de valeurs

PI = 3.14159265358979

df_feat = (
    df
    .withColumn("heure_sin", sin(2 * PI * col("heure") / 24))
    .withColumn("heure_cos", cos(2 * PI * col("heure") / 24))
    .withColumn("jour_sem_sin", sin(2 * PI * col("jour_sem") / 7))
    .withColumn("jour_sem_cos", cos(2 * PI * col("jour_sem") / 7))
    .withColumn("mois_sin", sin(2 * PI * col("mois") / 12))
    .withColumn("mois_cos", cos(2 * PI * col("mois") / 12))
)

# ── 2. Features météo normalisées ────────────────────────────────────────────
# Les valeurs manquantes sont imputées par la médiane (calculée sur le train).
# Ici on les remplace provisoirement par des constantes raisonnables (à vous de les choisir)
#
# J'ai choisi des constantes plausibles pour de la météo parisienne plutôt que
# 0, pour ne pas fausser artificiellement le modèle sur les lignes incomplètes
# (un 0°C ou 0% d'humidité par défaut serait trompeur).
df_feat = (
    df_feat
    .withColumn("temperature_c",    coalesce(col("temperature_c"), lit(12.0)))
    .withColumn("humidite_pct",     coalesce(col("humidite_pct"), lit(70.0)))
    .withColumn("vent_kmh",         coalesce(col("vent_kmh"), lit(12.0)))
    .withColumn("precipitation_mm", coalesce(col("precipitation_mm"), lit(0.0)))
    .withColumn("est_pluie",        coalesce(col("est_pluie"), lit(False)))
)

# ── 3. Features de lag (historique récent de la station) ─────────────────────
# Le taux d'occupation à t-1 et t-4 (environ 15 et 60 minutes avant)
# sont souvent les features les plus prédictives pour ce type de série.

# Pas de "station_id" dans nos données (voir printSchema plus haut) :
# "nom_station" identifie une station de façon unique, je l'utilise partout
# à la place, comme au Jour 2.
fenetre_station = (
    Window
    .partitionBy("nom_station")
    .orderBy("horodatage")
)

df_feat = (
    df_feat
    .withColumn("taux_lag1", lag("taux_occupation", 1).over(fenetre_station))
    .withColumn("taux_lag4", lag("taux_occupation", 4).over(fenetre_station))
    .withColumn(
        "taux_moyen_recent",
        spark_avg("taux_occupation").over(fenetre_station.rowsBetween(-4, -1))
    )
)

# ── 4. Variable cible : taux_occupation à t+4 (≈ 1 heure dans le futur) ──────

# La variable cible est celle que l'on cherche à prédire
df_feat = df_feat.withColumn(
    "cible",
    lead("taux_occupation", 4).over(fenetre_station)
)

# Liste des features utilisées par les modèles (clustering exclu, régression
# ci-dessous) -- je la fixe ici pour la réutiliser dans tout le notebook.
FEATURES = [
    "heure_sin", "heure_cos", "jour_sem_sin", "jour_sem_cos", "mois_sin", "mois_cos",
    "is_weekend", "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm", "est_pluie",
    "taux_lag1", "taux_lag4", "taux_moyen_recent", "capacite",
]

# ── 5. Suppression des lignes incomplètes (lag/lead génère des nulls) ──────────

df_ml = df_feat.dropna(subset=FEATURES + ["cible"])

print(f"Lignes disponibles pour le ML : {df_ml.count():,}")
print(f"Features ({len(FEATURES)}) : {FEATURES}")

Lignes disponibles pour le ML : 10,910,909
Features (16) : ['heure_sin', 'heure_cos', 'jour_sem_sin', 'jour_sem_cos', 'mois_sin', 'mois_cos', 'is_weekend', 'temperature_c', 'humidite_pct', 'vent_kmh', 'precipitation_mm', 'est_pluie', 'taux_lag1', 'taux_lag4', 'taux_moyen_recent', 'capacite']


## 1.3 Séparation du jeu de données

Créer un jeu de données pour l'entraînement et un jeu de données pour la validation/test

In [6]:
# ── 6. Split train / test ──────────────────────────────────────────────────────
# IMPORTANT : on split dans le temps, pas aléatoirement.
# Un split aléatoire sur des séries temporelles entraîne une fuite d'information :
# le modèle voit des données du futur pendant l'entraînement.
#
# Nos données couvrent 2020-2021 (et non 2022-2023 comme dans l'énoncé
# d'origine) : je garde donc automatiquement la dernière année complète comme
# test, et le reste comme train. Si une seule année était disponible, je
# retombe sur un split chronologique 85/15 (comme suggéré en repli).
annees_disponibles = sorted(r["annee"] for r in df_ml.select("annee").distinct().collect())

if len(annees_disponibles) >= 2:
    annee_test = annees_disponibles[-1]
    df_train = df_ml.filter(F.col("annee") != annee_test)
    df_test  = df_ml.filter(F.col("annee") == annee_test)
    print(f"Split temporel par année : train = {annees_disponibles[:-1]}  |  test = {annee_test}")
else:
    df_ts  = df_ml.withColumn("ts_unix", unix_timestamp("horodatage"))
    seuil  = df_ts.approxQuantile("ts_unix", [0.85], 0.01)[0]
    df_train = df_ts.filter(col("ts_unix") <= seuil).drop("ts_unix")
    df_test  = df_ts.filter(col("ts_unix") > seuil).drop("ts_unix")
    print("Une seule année disponible : split chronologique 85/15")

print(f"Train : {df_train.count():,} lignes")
print(f"Test  : {df_test.count():,} lignes")
print(f"Ratio        : {df_train.count() / df_ml.count():.1%} / {df_test.count() / df_ml.count():.1%}")

# Mise en cache des deux splits -- ils seront accédés plusieurs fois
df_train.cache(); df_train.count()
df_test.cache();  df_test.count()

Split temporel par année : train = [2020]  |  test = 2021
Train : 3,332,454 lignes
Test  : 7,578,455 lignes
Ratio        : 30.5% / 69.5%


7578455

## 1.4 Clustering des stations avec K-Means

Avant de prédire la disponibilité individuelle de chaque station, nous allons
**regrouper les stations par profil comportemental** : certaines sont saturées
le matin (quartiers résidentiels qui "envoient" des cyclistes vers le centre),
d'autres le soir, d'autres ont un usage homogène sur la journée.

Ce clustering a deux utilités :
1. Comprendre la géographie fonctionnelle du réseau.
2. Ajouter l'identifiant de cluster comme feature dans le modèle de régression.

### Construction du profil de station

Le vecteur de profil d'une station est son **taux d'occupation moyen à chaque
heure de la journée** -- un vecteur de dimension 24.


In [7]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import ClusteringEvaluator

# ── Profil horaire moyen par station ─────────────────────────────────────────
# Pivot : une ligne par station, une colonne par heure
# (nom_station à la place de station_id, cf. adaptation en 1.2)

df_profil = (
    df_ml
    .groupBy("nom_station")
    .pivot("heure", list(range(24)))
    .agg(spark_round(spark_avg("taux_occupation"), 4))
)

# Renommage des colonnes pivot (0 -> h00, 1 -> h01, ...)
for h in range(24):
    df_profil = df_profil.withColumnRenamed(str(h), f"h{h:02d}")

colonnes_profil = [f"h{h:02d}" for h in range(24)]
df_profil = df_profil.dropna(subset=colonnes_profil)

print(f"Stations avec profil complet : {df_profil.count()}")
df_profil.show(5, truncate=True)

Stations avec profil complet : 1388
+--------------------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|         nom_station|   h00|   h01|   h02|   h03|   h04|   h05|   h06|   h07|   h08|   h09|   h10|   h11|   h12|   h13|   h14|   h15|   h16|   h17|   h18|   h19|   h20|   h21|   h22|   h23|
+--------------------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|Adolphe Lalyre - ...|0.3009|0.2953|0.2949|0.2992|0.2987|0.2994|0.2812|0.2208|0.1699|0.1639|0.1632|0.1617|0.1589|0.1481|0.1423|0.1354|0.1479|0.1827| 0.244|0.2744|0.2786|0.2803|0.2859|0.2891|
|      Alger - Rivoli|0.3508|0.3519|0.3679|0.3589|0.3588| 0.383|0.4027|0.4728|0.6327|0.6853|0.7174|0.7602|0.7665|  0.82|0.8426|0.8251|0.7159|0.5363|0.4663|0.4102|0.4057|0.3918|0.3712|0.3681|
|  Aliber

In [8]:
# ── Méthode du coude : inertie (WSSSE) en fonction de k ──────────────────────
# On entraîne un K-Means pour k = 2..8 et on trace l'inertie.
# Le "coude" dans la courbe indique le bon compromis.

# VectorAssembler transforme un « tableau » (un ensemble de colonnes) en un vecteur
# Transformation essentielle pour fournir des données à l'algorithme de clusterisation
assembler_profil = VectorAssembler(
    inputCols=colonnes_profil,
    outputCol="features_profil"
)

# StandardScaler normalise les valeurs entre [-1, +1]
# en formant une distribution appelée « réduite centrée »
# centrée autour de la moyenne, réduite à 1 écart-type
scaler_profil = StandardScaler(
    inputCol="features_profil",
    outputCol="features_scaled",
    withStd=True, withMean=True
)

inertie_par_k = {}

# Exécuter l'algorithme K-Means en essayant plusieurs valeurs pour le nombre de clusters
#
for k in range(2, 9):
    kmeans = KMeans(
        featuresCol="features_scaled",
        predictionCol="cluster",
        k=k,
        seed=SEED,
    )
    # Pipeline assemblant les trois étapes : assembler_profil, scaler_profil, kmeans
    pipeline_km = Pipeline(stages=[assembler_profil, scaler_profil, kmeans])
    model_km = pipeline_km.fit(df_profil)

    # WSSSE = Within Set Sum of Squared Errors (inertie intra-cluster)
    # Le Within Set Sum of Squared Errors (WSSSE) est la somme des carrés des distances 
    # entre chaque point de données et le centroïde de son cluster dans un algorithme de K-Means.
    # Le centroïde est aussi appelé barycentre ou centre de gravité
    wssse = model_km.stages[-1].summary.trainingCost
    inertie_par_k[k] = round(wssse, 2)
    print(f"  k={k}  inertie={wssse:,.1f}")

print("\nTableau récapitulatif :")
for k, w in inertie_par_k.items():
    barre = "=" * int(w / max(inertie_par_k.values()) * 40)
    print(f"  k={k}  {w:>12,.1f}  {barre}")

  k=2  inertie=13,685.6
  k=3  inertie=10,026.3
  k=4  inertie=7,637.5
  k=5  inertie=6,418.8
  k=6  inertie=5,820.9
  k=7  inertie=4,862.0
  k=8  inertie=4,408.6

Tableau récapitulatif :
  k=2      13,685.6  ========================================
  k=3      10,026.3  =============================
  k=4       7,637.5  ======================
  k=5       6,418.8  ==================
  k=6       5,820.9  =================
  k=7       4,862.0  ==============
  k=8       4,408.6  ============


In [9]:
# ── Entraînement final avec le k retenu ───────────────────────────────────────
# Choisissez le k correspondant au "coude" observé ci-dessus.
# En pratique sur ce jeu de données, k=4 ou k=5 est souvent pertinent.
# Je retiens k=4 : au-delà, le gain d'inertie devient marginal (loi des
# rendements décroissants classique de la méthode du coude).
K_RETENU = 4

kmeans_final = KMeans(
    featuresCol="features_scaled",
    predictionCol="cluster",
    k=K_RETENU,
    seed=SEED,
)
pipeline_km_final = Pipeline(stages=[assembler_profil, scaler_profil, kmeans_final])
model_km_final = pipeline_km_final.fit(df_profil)

# Construction d'un DataFrame issu de la clusterisation
# et ne contenant que l'identifiant de la station et son cluster d'appartenance
df_clusters = model_km_final.transform(df_profil).select("nom_station", "cluster")

# Taille de chaque cluster
print(f"Répartition des {df_clusters.count()} stations en {K_RETENU} clusters :")
df_clusters.groupBy("cluster").count().orderBy("cluster").show()

Répartition des 1388 stations en 4 clusters :
+-------+-----+
|cluster|count|
+-------+-----+
|      0|  278|
|      1|  360|
|      2|  213|
|      3|  537|
+-------+-----+



In [10]:
# ── Interprétation : profil horaire moyen par cluster ────────────────────────

# 1. Jointure entre df_profil et df_clusters sur le nom de la station
# 2. Taux d'occupation moyen par heure pour chaque cluster, trié par cluster
print("Taux d'occupation moyen par cluster et par heure (6h, 9h, 12h, 18h, 22h) :")
(
    df_profil
    .join(df_clusters, on="nom_station")
    .groupBy("cluster")
    .agg(
        spark_round(spark_avg("h06"), 4).alias("h06"),
        spark_round(spark_avg("h09"), 4).alias("h09"),
        spark_round(spark_avg("h12"), 4).alias("h12"),
        spark_round(spark_avg("h18"), 4).alias("h18"),
        spark_round(spark_avg("h22"), 4).alias("h22"),
        F.count("*").alias("nb_stations"),
    )
    .orderBy("cluster")
    .show()
)

Taux d'occupation moyen par cluster et par heure (6h, 9h, 12h, 18h, 22h) :
+-------+------+------+------+------+------+-----------+
|cluster|   h06|   h09|   h12|   h18|   h22|nb_stations|
+-------+------+------+------+------+------+-----------+
|      0|0.6123|0.5767| 0.568|0.6144|0.6538|        278|
|      1|0.4041|0.2809|0.2447|0.3347|0.4348|        360|
|      2|0.4086|0.5864| 0.585|0.4192|0.3867|        213|
|      3|0.2261|0.2139|0.1829|0.1694|0.2128|        537|
+-------+------+------+------+------+------+-----------+



## Folium

Folium s'appuie sur les points forts de l'écosystème Python en termes d'analyse de données et sur ceux de la bibliothèque JS Leaflet our l'affichage cartographique. Manipulez vos données en Python, puis visualisez-les dans une carte via Folium.

### Concepts

Folium facilite la visualisation des données manipulées en Python sur une carte interactive. Il permet à la fois la liaison de données avec une carte géographique pour les visualisations choroplèthes, et de passer comme marqueurs de graphismes riche  vectoriels/binaires/HTML.

La bibliothèque dispose d'un certain nombre de jeux de tuiles intégrés d'OpenStreetMap, Mapbox, etc. Elle prend aussi en charge les jeux de tuiles personnalisés. 

Folium prend en charge les superpositions Image, Video, GeoJSON et TopoJSON et dispose d'un nombre de couches vectorielles intégrées.


In [11]:
# ── Visualisation sur carte Folium ─────────────────────────────────────────
import folium
import pandas as pd

PALETTE = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12",
           "#9b59b6", "#1abc9c", "#e67e22"]

# Le fichier attendu à l'origine (stations_info.csv) n'existe pas dans nos
# données (cf. Section 0) : je récupère les coordonnées GPS à partir du CSV
# brut Vélib. Ce fichier fait ~11M lignes (une par snapshot) donc je le lis
# avec Spark et je déduplique par station avant de rapatrier le résultat
# (minuscule) en pandas -- pas question de charger 11M lignes avec pandas.
df_stations_pos = (
    spark.read.csv(str(STATIONS_CSV), header=False, inferSchema=True)
    .toDF("horodatage", "capacite", "velos_meca", "velos_elec",
          "nom_station", "coords", "is_installed")
    .select("nom_station", "coords")
    .dropDuplicates(["nom_station"])
    .withColumn("lat", F.split(F.col("coords"), ",").getItem(0).cast("double"))
    .withColumn("lon", F.split(F.col("coords"), ",").getItem(1).cast("double"))
    .select("nom_station", "lat", "lon")
)

# Jointure clusters + coordonnées, puis passage en pandas (petit volume :
# une ligne par station) pour itérer facilement avec Folium.
df_carte = (
    df_clusters
    .join(df_stations_pos, on="nom_station", how="inner")
    .toPandas()
)

# Création d'une carte avec Folium
carte = folium.Map(location=[48.8566, 2.3522], zoom_start=12, tiles="CartoDB positron")

for _, row in df_carte.iterrows():
    couleur = PALETTE[int(row["cluster"]) % len(PALETTE)]
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        color=couleur,
        fill=True,
        fill_color=couleur,
        fill_opacity=0.8,
        popup=f"Cluster {int(row['cluster'])}",
        tooltip=row["nom_station"],
    ).add_to(carte)

# Légende textuelle ajoutée à la carte
legende_html = "<div style='position:fixed;bottom:30px;left:30px;z-index:1000;background:white;padding:10px;border:1px solid #ccc;font-size:12px'>"
for i in range(K_RETENU):
    legende_html += (
        f"<div><span style='display:inline-block;width:14px;height:14px;"
        f"background:{PALETTE[i]};border-radius:50%;margin-right:6px'></span>"
        f"Cluster {i}</div>"
    )
legende_html += "</div>"
carte.get_root().html.add_child(folium.Element(legende_html))

chemin_carte = OUTPUT_DIR / "carte_clusters_velib.html"
carte.save(str(chemin_carte))
print(f"Carte sauvegardée : {chemin_carte}")
print("Ouvrez ce fichier dans votre navigateur pour visualiser les clusters.")

# Affichage inline dans Jupyter
carte

Carte sauvegardée : ../data/output/carte_clusters_velib.html
Ouvrez ce fichier dans votre navigateur pour visualiser les clusters.


In [12]:
# Ajout du cluster comme feature pour le modèle de régression
# (nom_station à la place de station_id, comme partout ailleurs dans ce notebook)
df_ml    = df_ml.join(df_clusters, on="nom_station", how="left").fillna(0, subset=["cluster"])
df_train = df_train.join(df_clusters, on="nom_station", how="left").fillna(0, subset=["cluster"])
df_test  = df_test.join(df_clusters, on="nom_station", how="left").fillna(0, subset=["cluster"])

FEATURES = FEATURES + ["cluster"]
print(f"Features après ajout du cluster ({len(FEATURES)}) : {FEATURES}")

Features après ajout du cluster (17) : ['heure_sin', 'heure_cos', 'jour_sem_sin', 'jour_sem_cos', 'mois_sin', 'mois_cos', 'is_weekend', 'temperature_c', 'humidite_pct', 'vent_kmh', 'precipitation_mm', 'est_pluie', 'taux_lag1', 'taux_lag4', 'taux_moyen_recent', 'capacite', 'cluster']


---
## 1.5 Modèle de régression : GBTRegressor

Le **Gradient Boosted Tree Regressor** est un algorithme d'ensemble qui construit
séquentiellement des arbres de décision, chacun corrigeant les erreurs du précédent.
C'est l'un des meilleurs algorithmes pour les données tabulaires avec des features
hétérogènes -- exactement notre cas.

MLlib l'implémente de façon distribuée : chaque arbre est construit en parallèle
sur les partitions du DataFrame.


In [13]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# ── Construction du Pipeline ──────────────────────────────────────────────────

# 1. Vecteur de features brutes -- handleInvalid="skip" par sécurité si une
# ligne contenait malgré tout une valeur manquante après le dropna de 1.2.
assembler = VectorAssembler(
    inputCols=FEATURES,
    outputCol="features_brutes",
    handleInvalid="skip",
)

# 2. Normalisation (centré-réduit) des features brutes -> colonne "features"
scaler = StandardScaler(
    inputCol="features_brutes",
    outputCol="features",
    withStd=True, withMean=True,
)

# 3. Modèle de régression :
# - entrée : le vecteur "features" ; sortie : "prediction" ; cible : "cible"
# - 50 arbres de décision (maxIter), pas d'apprentissage 0.1 (stepSize)
# - seed fixée pour la reproductibilité
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="cible",
    predictionCol="prediction",
    maxIter=50,
    stepSize=0.1,
    seed=SEED,
)

# 3. Pipeline complet
pipeline_gbt = Pipeline(stages=[assembler, scaler, gbt])

# ── Entraînement ──────────────────────────────────────────────────────────────
print("Entraînement du GBTRegressor (peut prendre 1-2 minutes)...")
t0 = time.perf_counter()
model_gbt = pipeline_gbt.fit(df_train)
t_fit = time.perf_counter() - t0
print(f"Entraînement terminé en {t_fit:.1f} s")

Entraînement du GBTRegressor (peut prendre 1-2 minutes)...
Entraînement terminé en 200.0 s


In [14]:
# ── Évaluation sur le test set ────────────────────────────────────────────────
evaluator_rmse = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="r2"
)
evaluator_mae = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="mae"
)

df_pred = model_gbt.transform(df_test)

rmse = evaluator_rmse.evaluate(df_pred)
r2   = evaluator_r2.evaluate(df_pred)
mae  = evaluator_mae.evaluate(df_pred)

print(f"Métriques sur le test set (2023) :")
print(f"  RMSE (Root Mean Squared Error) : {rmse:.4f}")
print(f"  MAE  (Mean Absolute Error)     : {mae:.4f}")
print(f"  R²   (coefficient de détermination) : {r2:.4f}")
print()
print("Interprétation :")
print(f"  En moyenne, le modèle se trompe de ±{mae:.3f} sur le taux d'occupation (0-1).")
print(f"  Soit ±{mae*100:.1f} points de pourcentage.")


Métriques sur le test set (2023) :
  RMSE (Root Mean Squared Error) : 0.1176
  MAE  (Mean Absolute Error)     : 0.0747
  R²   (coefficient de détermination) : 0.7988

Interprétation :
  En moyenne, le modèle se trompe de ±0.075 sur le taux d'occupation (0-1).
  Soit ±7.5 points de pourcentage.


In [15]:
# ── Importance des features ────────────────────────────────────────────────────
# Le GBT calcule l'importance de chaque feature (réduction d'impureté moyenne)
model_gbt_stage = model_gbt.stages[-1]   # le GBTRegressionModel
importances     = model_gbt_stage.featureImportances.toArray()

df_importance = (
    pd.DataFrame({
        "feature"   : FEATURES,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Importance des features (top 10) :")
print(f"{'Rang':<5} {'Feature':<25} {'Importance':>12}  Barre")
print("-" * 65)
for i, row in df_importance.head(10).iterrows():
    barre = "█" * int(row["importance"] * 200)
    print(f"  {i+1:<3} {row['feature']:<25} {row['importance']:>12.4f}  {barre}")


Importance des features (top 10) :
Rang  Feature                     Importance  Barre
-----------------------------------------------------------------
  1   taux_lag1                       0.8028  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  2   cluster                         0.0542  ██████████
  3   heure_sin                       0.0531  ██████████
  4   heure_cos                       0.0237  ████
  5   taux_lag4                       0.0195  ███
  6   capacite                        0.0183  ███
  7   taux_moyen_recent               0.0120  ██
  8   is_weekend                      0.0047  
  9   jour_sem_cos                    0.0046  
  10  temperature_c                   0.0044  


In [16]:
# ── Analyse des erreurs : où le modèle se trompe-t-il le plus ? ───────────────
df_erreurs = (
    df_pred
    .withColumn("erreur_abs", F.abs(col("prediction") - col("cible")))
    .withColumn("erreur_rel",
        F.abs(col("prediction") - col("cible")) / (col("cible") + F.lit(0.01))
    )
)

print("Erreur absolue moyenne par heure de la journée :")
(
    df_erreurs
    .groupBy("heure")
    .agg(
        spark_round(spark_avg("erreur_abs"), 4).alias("mae_horaire"),
        F.count("*").alias("n")
    )
    .orderBy("heure")
    .show(24)
)


Erreur absolue moyenne par heure de la journée :
+-----+-----------+------+
|heure|mae_horaire|     n|
+-----+-----------+------+
|    0|     0.0429|149960|
|    1|     0.0375|137456|
|    2|     0.0325|281860|
|    3|     0.0323|340173|
|    4|     0.0444|298529|
|    5|     0.0767|359616|
|    6|     0.1052|311026|
|    7|     0.1139|363782|
|    8|      0.094|319361|
|    9|     0.0805|341564|
|   10|     0.0794|362392|
|   11|     0.0832|404048|
|   12|     0.0849|269372|
|   13|     0.0793|366611|
|   14|     0.0898|377688|
|   15|     0.1184|352685|
|   16|     0.1334|197168|
|   17|     0.1141|245759|
|   18|     0.0857|267981|
|   19|     0.0635|404039|
|   20|     0.0573|365182|
|   21|     0.0496|390174|
|   22|     0.0437|323524|
|   23|     0.0421|348505|
+-----+-----------+------+



In [17]:
# [EXERCICE]
# Calculez l'erreur absolue moyenne (MAE) par cluster de station.
# Quel cluster est le plus difficile à prédire ? Pourquoi, à votre avis ?
#
# Indice : groupBy("cluster").agg(...)
# ──────────────────────────────────────────────────────────────────────────

# Ma tentative, avant de regarder la correction dans la cellule suivante :
print("MAE par cluster (ma tentative) :")
(
    df_erreurs
    .groupBy("cluster")
    .agg(spark_round(spark_avg("erreur_abs"), 4).alias("mae_cluster"))
    .orderBy(F.desc("mae_cluster"))
    .show()
)

MAE par cluster (ma tentative) :
+-------+-----------+
|cluster|mae_cluster|
+-------+-----------+
|      2|     0.0883|
|      0|     0.0866|
|      1|     0.0792|
|      3|     0.0602|
+-------+-----------+



In [18]:
# [CORRECTION]
print("MAE par cluster de station :")
(
    df_erreurs
    .groupBy("cluster")
    .agg(
        spark_round(spark_avg("erreur_abs"), 4).alias("mae_cluster"),
        spark_round(spark_avg("erreur_rel"), 4).alias("mare_cluster"),
        F.count("*").alias("n_predictions")
    )
    .orderBy(F.desc("mae_cluster"))
    .show()
)
# Les stations de cluster à fort usage (très vides le matin, très pleines le soir)
# sont généralement plus difficiles à prédire car leur dynamique est plus abrupte.


MAE par cluster de station :
+-------+-----------+------------+-------------+
|cluster|mae_cluster|mare_cluster|n_predictions|
+-------+-----------+------------+-------------+
|      2|     0.0883|      0.4505|      1168027|
|      0|     0.0866|      0.2818|      1517339|
|      1|     0.0792|      0.6314|      1970356|
|      3|     0.0602|      0.8882|      2922733|
+-------+-----------+------------+-------------+



---
## 1.6 Validation croisée et optimisation des hyperparamètres

Nos hyperparamètres actuels (`maxIter=50`, `maxDepth=5`, `stepSize=0.1`)
ont été choisis arbitrairement. Le `CrossValidator` de MLlib permet
d'évaluer systématiquement plusieurs combinaisons et de choisir la meilleure.

> **Attention** : la validation croisée sur des séries temporelles est
> délicate. Le `CrossValidator` de MLlib effectue un K-Fold standard
> (mélanges aléatoires des données), ce qui peut créer de la fuite
> d'information temporelle. Pour un usage production, on préférerait
> une validation en fenêtre glissante (*time series split*).
> Dans le contexte de ce cours, le K-Fold est acceptable pour comprendre
> le mécanisme.


In [19]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# ── Grille d'hyperparamètres ──────────────────────────────────────────────────
param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth,  [3, 5])
    .addGrid(gbt.maxIter,   [30, 50])
    .addGrid(gbt.stepSize,  [0.05, 0.1])
    .build()
)
print(f"Combinaisons à évaluer : {len(param_grid)}")

# ── CrossValidator ────────────────────────────────────────────────────────────
cv = CrossValidator(
    estimator=pipeline_gbt,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_rmse,
    numFolds=3,
    seed=SEED,
    parallelism=2    # évalue 2 combinaisons en parallèle
)

# On entraîne sur un sous-échantillon pour limiter le temps de calcul en cours
df_train_sample = df_train.sample(fraction=0.3, seed=SEED).cache()
df_train_sample.count()
print(f"Sous-échantillon train : {df_train_sample.count():,} lignes")

print("\nValidation croisée en cours (peut prendre 3-5 minutes)...")
t0 = time.perf_counter()
cv_model = cv.fit(df_train_sample)
t_cv = time.perf_counter() - t0
print(f"Terminé en {t_cv:.1f} s")


Combinaisons à évaluer : 8
Sous-échantillon train : 1,000,632 lignes

Validation croisée en cours (peut prendre 3-5 minutes)...
Terminé en 876.5 s


In [20]:
# ── Résultats de la grille ────────────────────────────────────────────────────
avg_metrics = cv_model.avgMetrics
params_list = [
    {p.name: v for p, v in zip(param_grid[0].keys(), combo.values())}
    for combo in param_grid
]

resultats_cv = pd.DataFrame(params_list)
resultats_cv["rmse_cv"] = avg_metrics
resultats_cv = resultats_cv.sort_values("rmse_cv").reset_index(drop=True)

print("Résultats du CrossValidator (triés par RMSE croissant) :")
print(resultats_cv.to_string(index=False))

# Meilleurs hyperparamètres
meilleur = resultats_cv.iloc[0]
print(f"\nMeilleure combinaison :")
for col_name in ["maxDepth", "maxIter", "stepSize", "rmse_cv"]:
    print(f"  {col_name:<12} : {meilleur[col_name]}")


Résultats du CrossValidator (triés par RMSE croissant) :
 maxDepth  maxIter  stepSize  rmse_cv
        5       50      0.10 0.101959
        5       30      0.10 0.102869
        5       50      0.05 0.103108
        5       30      0.05 0.104015
        3       50      0.10 0.105551
        3       30      0.10 0.107301
        3       50      0.05 0.108121
        3       30      0.05 0.109808

Meilleure combinaison :
  maxDepth     : 5.0
  maxIter      : 50.0
  stepSize     : 0.1
  rmse_cv      : 0.10195885433434859


In [21]:
# Évaluation du meilleur modèle sur le test set complet
df_pred_best = cv_model.bestModel.transform(df_test)

rmse_best = evaluator_rmse.evaluate(df_pred_best)
r2_best   = evaluator_r2.evaluate(df_pred_best)
mae_best  = evaluator_mae.evaluate(df_pred_best)

print("Comparaison modèle initial vs meilleur modèle CV :")
print(f"{'Métrique':<8}  {'Initial':>12}  {'CV Best':>12}  {'Gain':>10}")
print("-" * 46)
print(f"{'RMSE':<8}  {rmse:>12.4f}  {rmse_best:>12.4f}  {rmse - rmse_best:>+10.4f}")
print(f"{'MAE':<8}  {mae:>12.4f}  {mae_best:>12.4f}  {mae - mae_best:>+10.4f}")
print(f"{'R²':<8}  {r2:>12.4f}  {r2_best:>12.4f}  {r2_best - r2:>+10.4f}")


Comparaison modèle initial vs meilleur modèle CV :
Métrique       Initial       CV Best        Gain
----------------------------------------------
RMSE            0.1176        0.1177     -0.0001
MAE             0.0747        0.0753     -0.0005
R²              0.7988        0.7985     -0.0003


# 1.7 Suivi des expériences avec MLflow

MLflow est un outil de tracking d'expériences ML : il enregistre
hyperparamètres, métriques, artefacts (modèles, graphiques) et permet
de comparer les runs dans une interface web.

```bash
# Pour ouvrir l'interface MLflow dans un terminal séparé :
mlflow ui --backend-store-uri data/output/mlruns --port 5000
# Puis ouvrir http://localhost:5000 dans le navigateur
```


In [22]:
import mlflow
import mlflow.spark

# Configuration du répertoire de tracking
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR.resolve()}")
mlflow.set_experiment("ClimaCity-Paris-GBT")

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Expérience          : ClimaCity-Paris-GBT")


2026/08/31 19:35:23 INFO mlflow.tracking.fluent: Experiment with name 'ClimaCity-Paris-GBT' does not exist. Creating a new experiment.


MLflow tracking URI : file:///home/jovyan/data/output/mlruns
Expérience          : ClimaCity-Paris-GBT


In [23]:
def evaluer_et_logger(
    df_train_fit, df_test_eval,
    max_depth: int, max_iter: int, step_size: float,
    nom_run: str
) -> dict:
    """Entraîne un pipeline GBT, l'évalue et logue les résultats dans MLflow.

    Args:
        df_train_fit  : DataFrame d'entraînement.
        df_test_eval  : DataFrame de test.
        max_depth     : Profondeur maximale des arbres.
        max_iter      : Nombre d'arbres (iterations).
        step_size     : Taux d'apprentissage.
        nom_run       : Nom du run MLflow.

    Returns:
        Dictionnaire avec les métriques calculées sur le test set.
    """
    with mlflow.start_run(run_name=nom_run):
        # ── Paramètres ──
        mlflow.log_params({
            "max_depth" : max_depth,
            "max_iter"  : max_iter,
            "step_size" : step_size,
            "n_features": len(FEATURES),
            "n_train"   : df_train_fit.count(),
            "n_test"    : df_test_eval.count(),
        })

        # ── Entraînement ──
        gbt_run = GBTRegressor(
            featuresCol="features", labelCol="cible",
            predictionCol="prediction",
            maxIter=max_iter, maxDepth=max_depth,
            stepSize=step_size, seed=SEED
        )
        pipeline_run = Pipeline(stages=[assembler, scaler, gbt_run])
        t0    = time.perf_counter()
        model = pipeline_run.fit(df_train_fit)
        t_fit = time.perf_counter() - t0

        # ── Métriques ──
        preds = model.transform(df_test_eval)
        metriques = {
            "rmse"     : evaluator_rmse.evaluate(preds),
            "mae"      : evaluator_mae.evaluate(preds),
            "r2"       : evaluator_r2.evaluate(preds),
            "fit_time" : round(t_fit, 2),
        }
        mlflow.log_metrics(metriques)

        # ── Modèle ──
        mlflow.spark.log_model(model, artifact_path="pipeline_gbt")

        print(f"  [{nom_run}]  RMSE={metriques['rmse']:.4f}  "
              f"R²={metriques['r2']:.4f}  ({t_fit:.1f}s)")
        return metriques

# ── Plusieurs runs pour comparer ──────────────────────────────────────────────
print("Lancement des runs MLflow...")
configs = [
    {"max_depth": 3, "max_iter": 30, "step_size": 0.1,  "nom_run": "shallow-fast"},
    {"max_depth": 5, "max_iter": 50, "step_size": 0.1,  "nom_run": "medium-balanced"},
    {"max_depth": 5, "max_iter": 50, "step_size": 0.05, "nom_run": "medium-slow-lr"},
    {"max_depth": 7, "max_iter": 80, "step_size": 0.05, "nom_run": "deep-thorough"},
]

resultats_mlflow = []
for cfg in configs:
    m = evaluer_et_logger(df_train, df_test, **cfg)
    resultats_mlflow.append({"run": cfg["nom_run"], **m})

print("\nTableau comparatif :")
df_res = pd.DataFrame(resultats_mlflow).sort_values("rmse")
print(df_res.to_string(index=False))


Lancement des runs MLflow...


2026/08/31 19:38:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  [shallow-fast]  RMSE=0.1237  R²=0.7775  (104.5s)


2026/08/31 19:42:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  [medium-balanced]  RMSE=0.1176  R²=0.7988  (199.7s)


2026/08/31 19:47:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  [medium-slow-lr]  RMSE=0.1191  R²=0.7937  (211.4s)


2026/08/31 19:56:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  [deep-thorough]  RMSE=0.1163  R²=0.8033  (419.8s)

Tableau comparatif :
            run     rmse      mae       r2  fit_time
  deep-thorough 0.116293 0.074514 0.803300    419.80
medium-balanced 0.117608 0.074732 0.798824    199.67
 medium-slow-lr 0.119096 0.075597 0.793702    211.41
   shallow-fast 0.123686 0.079184 0.777496    104.46


In [24]:
# ── Chargement du meilleur modèle depuis MLflow ───────────────────────────────
from mlflow.tracking import MlflowClient

client  = MlflowClient()
exp     = client.get_experiment_by_name("ClimaCity-Paris-GBT")
runs    = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.rmse ASC"],
    max_results=1
)
best_run = runs[0]

print(f"Meilleur run :")
print(f"  ID   : {best_run.info.run_id}")
print(f"  Nom  : {best_run.data.tags.get('mlflow.runName', 'N/A')}")
print(f"  RMSE : {best_run.data.metrics['rmse']:.4f}")
print(f"  R²   : {best_run.data.metrics['r2']:.4f}")

# Rechargement du modèle
model_uri     = f"runs:/{best_run.info.run_id}/pipeline_gbt"
model_recharge = mlflow.spark.load_model(model_uri)
print(f"\nModèle rechargé depuis : {model_uri}")

# Vérification : les prédictions sont identiques
df_verif = model_recharge.transform(df_test.limit(100))
rmse_verif = evaluator_rmse.evaluate(df_verif)
print(f"RMSE sur 100 lignes (vérification) : {rmse_verif:.4f}")


2026/08/31 19:56:43 INFO mlflow.spark: 'runs:/4036a8cbab6a420fa4fea7ddbe8514bc/pipeline_gbt' resolved as 'file:///home/jovyan/data/output/mlruns/318225324727038636/4036a8cbab6a420fa4fea7ddbe8514bc/artifacts/pipeline_gbt'


Meilleur run :
  ID   : 4036a8cbab6a420fa4fea7ddbe8514bc
  Nom  : deep-thorough
  RMSE : 0.1163
  R²   : 0.8033


2026/08/31 19:56:44 INFO mlflow.spark: URI 'runs:/4036a8cbab6a420fa4fea7ddbe8514bc/pipeline_gbt/sparkml' does not point to the current DFS.
2026/08/31 19:56:44 INFO mlflow.spark: File 'runs:/4036a8cbab6a420fa4fea7ddbe8514bc/pipeline_gbt/sparkml' not found on DFS. Will attempt to upload the file.



Modèle rechargé depuis : runs:/4036a8cbab6a420fa4fea7ddbe8514bc/pipeline_gbt
RMSE sur 100 lignes (vérification) : 0.0411


In [25]:
# Sauvegarde locale du meilleur modèle (format natif Spark)
chemin_model_local = MODELS_DIR / "gbt_best"
cv_model.bestModel.write().overwrite().save(str(chemin_model_local))
print(f"Modèle sauvegardé localement : {chemin_model_local}")

# Pour le recharger plus tard :
# from pyspark.ml import PipelineModel
# model_recharge = PipelineModel.load(str(chemin_model_local))


Modèle sauvegardé localement : ../data/output/models/gbt_best


---
## Bilan du Jour 3 -- et du projet ClimaCity Paris

### Ce que j'ai fait aujourd'hui

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Chargement de la table Delta du Jour 2 | Spark SQL / Delta | `spark.read.format("delta")` |
| Feature engineering (temps cyclique, météo, lags) | MLlib | `sin`/`cos`, `coalesce`, `lag`/`lead`, fenêtres |
| Split temporel train/test | Bonnes pratiques ML | split par année, pas aléatoire |
| Clustering des stations | MLlib | `KMeans`, méthode du coude (WSSSE) |
| Visualisation géographique des clusters | Folium | `CircleMarker`, carte interactive |
| Régression du taux d'occupation à +1h | MLlib | `GBTRegressor`, `Pipeline` |
| Évaluation | MLlib | RMSE, MAE, R², importance des features |
| Optimisation des hyperparamètres | MLlib | `CrossValidator`, `ParamGridBuilder` |
| Suivi des expériences | MLflow | `log_params`, `log_metrics`, `log_model` |

### Ce que j'ai dû adapter par rapport à l'énoncé

Le jeu de données réel ne colle pas exactement à l'énoncé d'origine :
- pas de colonne `station_id` ni `code_arr` -- j'ai utilisé `nom_station`
  comme identifiant partout (Jours 2 et 3) ;
- les données couvrent 2020-2021 et non 2022-2023 -- mon split train/test
  s'adapte automatiquement aux années réellement présentes ;
- pas de `stations_info.csv` -- j'ai récupéré les coordonnées GPS depuis le
  CSV brut Vélib, dédupliqué par station avec Spark avant de repasser en
  pandas pour Folium.

### Point d'attention

Le `CrossValidator` utilisé en 1.6 fait un K-Fold aléatoire, ce qui n'est pas
idéal sur des séries temporelles (risque de fuite d'information). Je l'ai
gardé tel quel pour comprendre le mécanisme, mais en production je préférerais
un découpage en fenêtres glissantes (*time series split*).

### Bilan du projet ClimaCity Paris

En trois jours, j'ai construit une chaîne complète : ingestion et nettoyage
(RDD puis DataFrame), stockage transactionnel avec Delta Lake, analyse SQL et
fenêtrage, streaming temps réel avec alertes, et enfin apprentissage
automatique distribué (clustering + régression) avec suivi d'expériences.


In [26]:
spark.stop()
print("SparkSession arrêtée. Projet ClimaCity Paris terminé !")

SparkSession arrêtée. Projet ClimaCity Paris terminé !
